# Carte sémantique d'une liste d'URLs (POC)

**Ce que fait ce notebook** : à partir d'une **liste d'URLs**, il télécharge les pages (de façon robuste : plusieurs méthodes, plusieurs passes, cache), en extrait et nettoie le texte, calcule le *sens* de chaque article (vecteurs bge-m3),
regroupe les articles en **thèmes**, nomme chaque thème et dessine une **carte 2D interactive** (clic = fiche de l'article, recherche, couleurs par thème / site / langue).
Il liste aussi les paires d'articles les plus proches en sens (pistes de chevauchement ou de cannibalisation).

**Comment l'utiliser** : poser à côté du notebook (ou téléverser sur Colab) la liste d'URLs : l'**Excel du projet** (`PetCare - Article URLs.xlsx`, une feuille par marque), un CSV (colonne `url`) ou un fichier texte (une URL par ligne) ; indiquer son nom dans `FICHIER_URLS` (section 1), puis `Exécuter tout`.
**Aucun zip n'est nécessaire.** Les étapes 1 (téléchargement, nettoyage) et 2 (vecteurs) du projet Mars sont intégrées ici. Pour les URLs dont le texte est déjà connu (fichier `corpus_mars_avec_texte.csv`), le notebook ne télécharge rien ; les autres sont téléchargées.
Sur Colab : téléverser le notebook, l'Excel et le fichier de textes dans `/content` ; un GPU (T4) accélère beaucoup la vectorisation. **Le téléchargement complet de 685 URLs prend de l'ordre d'une demi-heure ou plus** (pauses entre les requêtes) ; il est mis en cache.
**Résultats** : dossier `outputs/carte_urls/` : `carte_urls.html` (la carte, autonome), `articles.csv`, `themes.csv`, `paires_proches.csv`, `pages_en_echec.csv` (avec la raison de chaque page écartée), `lignes_retirees.csv`, et le cache `html/`.

**À savoir** : certains sites bloquent les téléchargements automatiques ou se construisent en JavaScript. Le navigateur de secours (`UTILISER_NAVIGATEUR`) aide, mais n'est pas garanti ; les pages perdues sont listées avec leur raison, pas ignorées en silence.
Si le CSV contient une colonne `texte` (et `titre`), ces textes sont utilisés **sans téléchargement** : utile pour un export de CMS ou pour les sites qui refusent les accès automatiques.
Respecter les conditions d'utilisation des sites. Les noms de thèmes fonctionnent le mieux sur des textes en anglais (la vectorisation, elle, est multilingue).

## 0. Installation, chemins et imports
**Ce que ça fait** : installe les paquets (sans effet s'ils sont déjà là) et fixe les dossiers, comme dans le notebook principal. `trafilatura` extrait le texte principal d'une page web (sans menus ni publicités).

In [ ]:
import importlib.util, subprocess, sys
PAQUETS = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "umap": "umap-learn", "plotly": "plotly", "sentence_transformers": "sentence-transformers", "trafilatura": "trafilatura", "requests": "requests", "bs4": "beautifulsoup4", "lxml": "lxml", "curl_cffi": "curl_cffi", "openpyxl": "openpyxl"}   # module Python -> nom du paquet à installer
manquants = [paquet for module, paquet in PAQUETS.items() if importlib.util.find_spec(module) is None]
if manquants:                      # on n'installe que ce qui manque (en local avec le venv : rien ; sur Colab : peu de choses)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
print("paquets installés à l'instant :", manquants or "aucun, tout était déjà là")

import os, sys
IN_COLAB = "google.colab" in sys.modules
RACINE = ".." if os.path.isdir("../data") and not os.path.isdir("data") else "."   # lancé depuis notebooks/ du projet, ou depuis un dossier autonome
DATA_DIR = "/content/data" if IN_COLAB else f"{RACINE}/data"
OUT_DIR = "/content/outputs" if IN_COLAB else f"{RACINE}/outputs"
DOSSIERS_ENTREE = [".", DATA_DIR, "/content", "/content/drive/MyDrive"]   # où chercher le fichier d'URLs (dossier courant d'abord)
os.makedirs(f"{OUT_DIR}/carte_urls", exist_ok=True)
print("Colab" if IN_COLAB else "Local", "| Python :", sys.executable, "| sorties :", f"{OUT_DIR}/carte_urls")

import io, glob, re, json, html, time, os, random, hashlib, shutil, subprocess
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse, urljoin
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import umap
from plotly.offline import get_plotlyjs
from scipy.spatial import ConvexHull
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.manifold import trustworthiness
from sklearn.metrics import silhouette_samples
try:
    import trafilatura
except ImportError:
    trafilatura = None
    print("!! trafilatura absent : l'extracteur de secours (BeautifulSoup) sera utilisé")
try:
    from curl_cffi import requests as curl_requests
except ImportError:
    curl_requests = None
    print("!! curl_cffi absent : la méthode de téléchargement n° 2 est ignorée")

## 1. Paramètres
Tout ce qu'on peut régler est ici, avec son rôle et l'effet d'un changement.

In [ ]:
LISTE_URLS = []            # liste d'URLs collée ici (prioritaire sur le fichier) ; [] = lire le fichier ci-dessous
FICHIER_URLS = "PetCare - Article URLs.xlsx"   # fichier d'URLs : .xlsx (une feuille par marque, URLs en colonne A), .csv (colonne « url ») ou .txt (une URL par ligne)
FICHIER_TEXTES = "corpus_mars_avec_texte.csv"    # (facultatif) textes déjà récupérés : CSV avec colonnes url, titre, texte ; pas de téléchargement pour les URLs qu'il contient (None = ne rien utiliser)
N_MAX_PAR_MARQUE = None    # Excel : ex. 20 = garde les 20 premières URLs de chaque feuille (test rapide) ; None = toutes
N_MAX_URLS = None          # ex. 100 pour tester sur les 100 premières URLs ; None = toutes
DELAI = 1.5                # secondes entre deux téléchargements à la 1re passe (les passes suivantes sont plus lentes : 2,5 fois, puis 4 fois) ; ne pas descendre sous 0,5
PASSES = 3                 # passes de téléchargement avant le navigateur de secours (plus = plus de pages récupérées, mais plus long)
PAUSE_PASSES = 60          # secondes d'attente entre deux passes (laisse les protections anti-robot se calmer)
TIMEOUT = 25               # secondes avant d'abandonner un téléchargement
UTILISER_NAVIGATEUR = IN_COLAB   # dernier recours pour les pages bloquées : vrai navigateur (Playwright) ; True sur Colab, False en local (installation lourde)
LANGUES = ["en"]           # langues gardées (détectées sur le texte : "en", "es", "fr") ; None = toutes ; une langue non reconnue (« ? ») est écartée si le filtre est actif
MIN_MOTS = 100             # un article avec moins de mots utiles est marqué « court » (on lui ajoute sa description, il reste dans le corpus)
PAGES_MIN_REPETITION = 3   # nettoyage : une ligne d'au moins MOTS_MIN_LIGNE_REPETEE mots présente dans 3 pages d'une marque est un bloc répété (menu, promo) et est retirée
PAGES_MIN_REPETITION_LISTE = 5   # idem pour une puce de liste (seuil plus haut : les listes se répètent plus légitimement)
MOTS_MIN_LIGNE_REPETEE = 4 # nettoyage : une ligne plus courte n'est jamais considérée comme bloc répété
LIGNES_OUVERTURE = 2       # nettoyage : un teaser répété n'est gardé que s'il est dans les 2 premières lignes d'une page

MODELE = "BAAI/bge-m3"     # modèle de vectorisation (multilingue) ; en changer revient à tout recalculer
MAX_TOKENS = 8192          # longueur maximale lue par article : 8192 = article entier ; 512 va beaucoup plus vite sur CPU mais ne lit que le début
TAILLE_LOT = 8             # articles vectorisés à la fois sur GPU (baisse tout seul si la mémoire manque ; sur CPU : 2)
CALCULER_PASSAGES = True   # vectorise aussi des passages de texte (utile pour la cannibalisation) ; False = plus rapide
TAILLE_PASSAGE = 200       # mots par passage (petit = recoupements plus précis, mais plus de passages à vectoriser)
K = None                   # nombre de thèmes ; None = automatique : racine de (nombre d'articles / 2), entre 3 et 20 (règle simple, pas optimisée)
GRAINE = 42                # graine unique (K-means, UMAP) : la changer change légèrement les résultats
N_INIT = 20                # K-means : nombre de départs aléatoires, on garde le meilleur
MAX_ITER = 300             # K-means : itérations maximum par départ

N_CANDIDATS = 80           # noms de thèmes : expressions candidates gardées par thème (plus = plus de choix, plus lent)
COUVERTURE_MIN = 0.15      # une expression candidate doit apparaître dans >= 15 % des articles du thème
POIDS_DISTINCTION = 0.5    # pénalité si l'expression ressemble aussi à un autre thème (haut = noms plus distincts)
POIDS_COUVERTURE = 0.15    # bonus si l'expression est dans beaucoup d'articles du thème (haut = noms plus représentatifs)
N_MOTS_CLES = 8            # mots-clés listés par thème dans le tableau des thèmes
MOTS_VIDES_SUPPL = set()   # mots à ignorer en plus pour nommer les thèmes (ex. noms de marques, mots du site) ; les noms de domaine sont ajoutés tout seuls

UMAP_VOISINS = 15          # UMAP : voisins considérés (petit = structure locale, grand = vue d'ensemble)
UMAP_MIN_DIST = 0.1        # UMAP : écart minimal entre points (petit = groupes plus serrés)
N_VOISINS_CONFIANCE = 10   # voisins utilisés pour la « fidélité » de la carte (trustworthiness, 1 = parfait)
PART_ZONE = 0.75           # part des articles d'un thème (les plus centraux) dans sa zone colorée
ZONE_MAX = 0.08            # pas de zone si elle couvre plus de 8 % de la carte (thème trop dispersé)
PAIRES_AFFICHEES = 30      # nombre de paires d'articles les plus proches en sens, listées à la fin

## 2. Lecture de la liste d'URLs et des textes déjà connus
**Ce que ça fait** : lit la liste d'URLs (collée, CSV, texte ou **Excel**), garde les adresses `http(s)`, supprime les doublons en conservant l'ordre.
Pour un Excel, **chaque feuille est une marque** (le nom de la feuille devient la marque). Si un fichier de textes est présent (`FICHIER_TEXTES`, colonnes `url`, `titre`, `texte`),
les URLs qu'il contient **ne sont pas téléchargées** : leur texte vient du fichier. Cela évite de dépendre des sites ; ces textes ne sont pas retouchés par le nettoyage de la section 3.

In [ ]:
def lire_csv_textes(chemin):
    "{url: page} pour un CSV qui contient une colonne « texte » (et éventuellement titre, description, langue)."
    t = pd.read_csv(chemin).dropna(subset=["url", "texte"])
    return {r["url"]: {"url": r["url"], "titre": str(r.get("titre", "") or ""), "description": str(r.get("description", "") or ""),
                       "texte": str(r["texte"]), "langue": str(r.get("langue", "") or "?")} for _, r in t.iterrows()}

def trouver(nom_fichier):
    "Premier fichier qui correspond dans DOSSIERS_ENTREE (None s'il n'y en a pas)."
    trouves = sorted(f for d in DOSSIERS_ENTREE for f in glob.glob(f"{d}/{nom_fichier}"))
    return trouves[0] if trouves else None

def lire_urls():
    "(URLs sans doublons, textes déjà connus {url: page}, marque de chaque URL {url: feuille Excel})."
    fournis, marque_feuille = {}, {}
    chemin_textes = trouver(FICHIER_TEXTES) if FICHIER_TEXTES else None
    if chemin_textes:                                      # textes déjà récupérés (ex. par l'équipe) : pas de téléchargement pour ces URLs
        fournis = lire_csv_textes(chemin_textes)
    chemin = None if LISTE_URLS else trouver(FICHIER_URLS)
    if LISTE_URLS:
        urls = list(LISTE_URLS)
    elif chemin is None:
        raise FileNotFoundError(f"{FICHIER_URLS} introuvable dans {DOSSIERS_ENTREE} : le déposer là ou remplir LISTE_URLS")
    elif chemin.endswith(".xlsx"):                         # Excel : une feuille par marque, URLs en colonne A
        urls = []
        for nom_feuille, t in pd.read_excel(chemin, sheet_name=None, header=None).items():
            liste = [str(v).strip() for v in t.iloc[:, 0].dropna() if str(v).strip().startswith("http")][:N_MAX_PAR_MARQUE]
            urls += liste
            marque_feuille.update({u: nom_feuille for u in liste})
    elif chemin.endswith(".csv"):
        t = pd.read_csv(chemin)
        urls = t["url" if "url" in t.columns else t.columns[0]].dropna().astype(str).tolist()
        if "texte" in t.columns:                           # le CSV d'URLs contient lui-même les textes
            fournis = {**fournis, **lire_csv_textes(chemin)}
    else:
        urls = [l.strip() for l in open(chemin, encoding="utf-8") if l.strip() and not l.startswith("#")]
    urls = [u.strip() for u in urls if u.strip().startswith("http")]
    return list(dict.fromkeys(urls))[:N_MAX_URLS], fournis, marque_feuille

urls, fournis, marque_feuille = lire_urls()
print(len(urls), "URLs | texte déjà connu (pas de téléchargement) :", sum(u in fournis for u in urls),
      "| à télécharger :", sum(u not in fournis for u in urls), "| marques :", sorted(set(marque_feuille.values())) or "(une par site)")

## 3. Téléchargement robuste, extraction et nettoyage du texte
**Ce que ça fait** : reprend l'étape 1 du projet Mars (code de `mars_step1_one_cell.py`, traduit et découpé). Pour chaque URL **sans texte déjà connu** :
1. télécharger la page avec `requests`, puis `curl_cffi` (qui imite le Chrome d'un vrai navigateur) ; plusieurs passes avec pauses ; les pages anti-robot sont détectées et jamais gardées ;
2. en dernier recours, un navigateur sans écran (Playwright), si `UTILISER_NAVIGATEUR = True` ;
3. extraire le texte principal, retirer les bandeaux, signatures et blocs répétés **marque par marque**, écarter doublons, pages liste/actualité et langues non voulues.

Les pages téléchargées sont **gardées en cache** (`outputs/carte_urls/html/`) : relancer le notebook ne retélécharge rien. **Paramètres qui comptent** : `DELAI`, `PASSES`, `PAUSE_PASSES`, `UTILISER_NAVIGATEUR`, `LANGUES`, `MIN_MOTS`.

### 3.1 Outils de base
**Ce que ça fait** : l'identité envoyée aux sites (celle d'un Chrome ordinaire) et `page_bloquee`, qui reconnaît une page anti-robot ou une page d'erreur servie avec le code 200 (une vraie page d'article fait plus de 100 Ko, une page de blocage est petite).

In [ ]:
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
ENTETES = {"User-Agent": UA, "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
           "Accept-Language": "en-US,en;q=0.9", "Upgrade-Insecure-Requests": "1"}
MARQUEURS_BLOCAGE = ["access denied", "403 forbidden", "just a moment", "attention required", "request unsuccessful", "captcha",
                     "are you a robot", "pardon our interruption", "bot detection", "unusual traffic", "request blocked"]
TITRE_ERREUR = re.compile(r"^(error page|error|page not found|not found|404\b.*|oops.*|page unavailable)( \|.*| -.*)?$", re.IGNORECASE)

def norm(texte):
    "Espaces multiples réduits à un seul."
    return re.sub(r"\s+", " ", str(texte)).strip()

def parties_chemin(url):
    return [p for p in urlparse(url).path.split("/") if p]

def page_bloquee(html):
    "Vrai pour une page anti-robot ou une page d'erreur : à ne jamais garder comme article."
    if not html or len(html) < 500:
        return True
    debut = html[:20000].lower()
    m = re.search(r"<title[^>]*>(.*?)</title>", debut, re.S)
    titre = m.group(1) if m else ""
    if len(html) < 50000 and any(k in titre for k in MARQUEURS_BLOCAGE):
        return True
    if TITRE_ERREUR.search(norm(titre)):
        return True                      # page d'erreur d'un site servie avec le code 200 (ex. article supprimé)
    return len(html) < 5000 and any(k in debut for k in MARQUEURS_BLOCAGE)

### 3.2 Les deux méthodes de téléchargement
**Ce que ça fait** : `telecharger(url)` essaie `requests`, puis `curl_cffi` (s'il est installé). Il s'arrête tout de suite sur une erreur 404/410 (la page n'existe pas). Il renvoie la page, la méthode qui a marché, ou la raison de l'échec.

In [ ]:
session = requests.Session()

def decoder(contenu, declare):
    "Texte de la page : encodage déclaré si valide, sinon UTF-8."
    for encodage in [declare, "utf-8"]:
        if encodage:
            try:
                return contenu.decode(encodage)
            except (LookupError, UnicodeDecodeError):
                pass
    return contenu.decode("utf-8", errors="replace")

def via_requests(url):
    r = session.get(url, headers=ENTETES, timeout=TIMEOUT, allow_redirects=True)
    return r.status_code, decoder(r.content, r.encoding if "charset" in r.headers.get("content-type", "") else "")

def via_curl(url):
    r = curl_requests.get(url, impersonate="chrome", timeout=TIMEOUT, allow_redirects=True)
    return r.status_code, decoder(r.content, "")

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
METHODES = [("requests", via_requests)] + ([("curl_cffi", via_curl)] if curl_requests else [])

def telecharger(url):
    "Renvoie (html, méthode, erreur) ; html vaut None si toutes les méthodes ont échoué."
    erreurs = []
    for nom, fonction in METHODES:
        try:
            statut, html = fonction(url)
        except Exception as e:
            erreurs.append(f"{nom} : {type(e).__name__}")
            continue
        if statut == 200 and not page_bloquee(html):
            return html, nom, ""
        erreurs.append(f"{nom} : HTTP {statut}" + (" (page de blocage)" if statut == 200 else ""))
        if statut in (404, 410):
            break                        # la page n'existe pas : inutile d'insister
    return None, "", " ; ".join(erreurs)

### 3.3 Cache et passes successives
**Ce que ça fait** : une page déjà enregistrée dans `outputs/carte_urls/html/` n'est pas retéléchargée (une page de blocage trouvée en cache est supprimée). Les pages en échec sont retentées lors de `PASSES` passes, avec une pause de `PAUSE_PASSES` secondes et un délai de plus en plus long : les sites tolèrent mieux un rythme lent.

In [ ]:
DOSSIER_HTML = f"{OUT_DIR}/carte_urls/html"
os.makedirs(DOSSIER_HTML, exist_ok=True)

def chemin_cache(url):
    "Fichier de la page en cache : dernier morceau de l'URL + 8 caractères de hachage (jamais deux URLs sur le même fichier)."
    slug = re.sub(r"[^A-Za-z0-9._-]", "_", (parties_chemin(url) or ["index"])[-1])[:80]
    return Path(DOSSIER_HTML) / f"{slug}_{hashlib.md5(url.encode()).hexdigest()[:8]}.html"

def en_cache(url):
    "Vrai si une vraie page est déjà enregistrée."
    f = chemin_cache(url)
    if f.exists() and page_bloquee(f.read_text(encoding="utf-8", errors="replace")):
        f.unlink()
    return f.exists()

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
def telecharger_tout(urls):
    "Télécharge les pages absentes du cache. Renvoie {url: dernière erreur} des pages encore manquantes."
    erreurs = {}
    for p in range(PASSES):
        reste = [u for u in urls if not en_cache(u) and "HTTP 404" not in erreurs.get(u, "") and "HTTP 410" not in erreurs.get(u, "")]
        if not reste:
            break
        if p > 0:
            print(f"{len(reste)} pages manquantes : pause de {PAUSE_PASSES} s avant la passe {p + 1}")
            time.sleep(PAUSE_PASSES)
        delai = DELAI * (1 + 1.5 * p)
        print(f"== passe {p + 1}/{PASSES} : {len(reste)} pages, {delai:.1f} s entre deux requêtes")
        for i, url in enumerate(reste, 1):
            html, methode, erreur = telecharger(url)
            if html is not None:
                chemin_cache(url).write_text(html, encoding="utf-8")
                erreurs.pop(url, None)
            else:
                erreurs[url] = erreur
            if i % 25 == 0 or i == len(reste):
                print(f"   {i}/{len(reste)} traitées")
            time.sleep(delai + random.uniform(0, 1))
    return {u: e for u, e in erreurs.items() if not en_cache(u)}

### 3.4 Navigateur de secours (facultatif)
**Ce que ça fait** : pour les pages que les deux méthodes n'ont pas pu avoir, ouvre chaque page dans un vrai navigateur sans écran (Playwright + Chromium, 1 à 2 minutes d'installation). Il tourne dans un processus séparé (évite les conflits avec Jupyter). **Désactivé en local par défaut** (`UTILISER_NAVIGATEUR`), activé sur Colab. Non testé ici.

In [ ]:
SCRIPT_NAVIGATEUR = r"""
import json, sys, time
from playwright.sync_api import sync_playwright
travaux = json.load(open(sys.argv[1]))
with sync_playwright() as p:
    navigateur = p.chromium.launch(args=["--disable-blink-features=AutomationControlled"])
    contexte = navigateur.new_context(user_agent=sys.argv[2], locale="en-US", viewport={"width": 1366, "height": 900})
    page = contexte.new_page()
    for t in travaux:
        statut, erreur = 0, ""
        try:
            reponse = page.goto(t["url"], wait_until="domcontentloaded", timeout=45000)
            statut = reponse.status if reponse else 0
            page.wait_for_timeout(5000)
            open(t["sortie"], "w", encoding="utf-8").write(page.content())
        except Exception as e:
            erreur = str(e)[:150]
        print(json.dumps({"url": t["url"], "statut": statut, "erreur": erreur}), flush=True)
        time.sleep(3)
    navigateur.close()
"""

**Ce que ça fait** : le navigateur de secours en lui-même (installation, puis ouverture de chaque page manquante).

In [ ]:
def installer_navigateur():
    "Installe Playwright et Chromium ; renvoie True si tout s'est bien passé."
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"]).returncode != 0:
        return False
    for options in (["--with-deps", "chromium"], ["chromium"]):           # avec les bibliothèques système (Colab), sinon sans
        if subprocess.run([sys.executable, "-m", "playwright", "install", *options], capture_output=True).returncode == 0:
            return True
    return False

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
def navigateur_secours(erreurs):
    "Retente avec un vrai navigateur les pages encore manquantes (hors 404/410). Renvoie les erreurs restantes."
    a_faire = [u for u, e in erreurs.items() if "HTTP 404" not in e and "HTTP 410" not in e]
    if not a_faire or not UTILISER_NAVIGATEUR:
        return erreurs
    print(f"{len(a_faire)} pages manquantes : installation du navigateur...")
    if not installer_navigateur():
        print("!! navigateur non installable : étape ignorée")
        return erreurs
    tmp = Path(OUT_DIR) / "carte_urls" / "_navigateur"
    tmp.mkdir(exist_ok=True)
    travaux = [{"url": u, "sortie": str(tmp / chemin_cache(u).name)} for u in a_faire]
    (tmp / "travaux.json").write_text(json.dumps(travaux))
    (tmp / "recuperer.py").write_text(SCRIPT_NAVIGATEUR)
    proc = subprocess.Popen([sys.executable, str(tmp / "recuperer.py"), str(tmp / "travaux.json"), UA], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ligne in proc.stdout:
        try:
            res = json.loads(ligne)
        except json.JSONDecodeError:
            continue
        source = tmp / chemin_cache(res["url"]).name
        html = source.read_text(encoding="utf-8") if source.exists() else ""
        if html and not page_bloquee(html):
            shutil.copy(source, chemin_cache(res["url"]))
            erreurs.pop(res["url"], None)
        else:
            erreurs[res["url"]] += f" ; navigateur : HTTP {res['statut']} {res['erreur']}".rstrip()
    proc.wait()
    return erreurs

### 3.5 Lancement du téléchargement
**Ce que ça fait** : télécharge (avec le cache) toutes les URLs sans texte déjà connu, puis passe au navigateur de secours si besoin. **C'est l'étape longue** : compter environ `DELAI` + 1 seconde par page à chaque passe. Pour un essai, régler `N_MAX_URLS` ou `N_MAX_PAR_MARQUE`.

In [ ]:
a_telecharger = [u for u in urls if u not in fournis]
print(f"{len(urls)} URLs | texte déjà connu : {len(urls) - len(a_telecharger)} | à télécharger : {len(a_telecharger)} dont {sum(not en_cache(u) for u in a_telecharger)} absentes du cache")
erreurs = telecharger_tout(a_telecharger)
erreurs = navigateur_secours(erreurs)
print(f"\n{len(erreurs)} pages impossibles à télécharger")
for u, e in list(erreurs.items())[:10]:
    print("  ", u, "->", e)

### 3.6 Extraction du texte principal
**Ce que ça fait** : `extraire(html, url)` renvoie titre, description, adresse canonique, texte principal et nombre de liens internes. Le texte vient de `trafilatura` ; si l'extraction donne moins de `MIN_MOTS` mots et que l'extracteur de secours (`BeautifulSoup`, balises `p`, `h1-h4`, `li`) en trouve plus, on prend le sien.
`detecter_langue` compte des mots courants anglais / espagnols / français (pas de bibliothèque de plus) et renvoie `?` si le texte est trop court ou ambigu.

In [ ]:
MOTS_COURANTS = {
    "en": {"the", "and", "is", "to", "of", "your", "you", "for", "with", "that", "it", "are"},
    "es": {"el", "la", "de", "que", "y", "los", "las", "para", "con", "su", "es", "una"},
    "fr": {"le", "la", "les", "de", "des", "et", "est", "pour", "votre", "vous", "une", "du"}}

def racine_contenu(soupe):
    "Zone principale de la page : on retire menus, pieds de page et scripts."
    for balise in soupe(["script", "style", "nav", "header", "footer", "aside", "form", "noscript"]):
        balise.decompose()
    return soupe.find("article") or soupe.find("main") or soupe.body

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
def texte_bs4(racine):
    "Texte de secours : titres, paragraphes et puces de la zone principale, une ligne par bloc."
    if racine is None:
        return ""
    blocs = []
    for el in racine.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
        if el.find_parent("a") is not None:                 # carte d'article liée : pas du contenu
            continue
        if el.name == "li" and el.find(["p", "li"]):        # le texte imbriqué est pris dans les enfants
            continue
        blocs.append(el.get_text(" ", strip=True))
    return "\n".join(b for b in blocs if b)

def detecter_langue(texte):
    mots = re.findall("[a-zà-ÿ]+", texte.lower())[:600]
    if len(mots) < 20:
        return "?"
    scores = {langue: sum(m in courants for m in mots) for langue, courants in MOTS_COURANTS.items()}
    meilleure = max(scores, key=scores.get)
    return meilleure if scores[meilleure] >= 5 else "?"

**Ce que ça fait** : `extraire` assemble le tout pour une page (métadonnées, texte principal, liens internes).

In [ ]:
def extraire(html, url):
    "Titre, description, adresse canonique, texte principal, nombre de liens internes d'une page."
    soupe = BeautifulSoup(html, "lxml")
    def meta(*noms):
        for nom in noms:
            balise = soupe.find("meta", attrs={"property": nom}) or soupe.find("meta", attrs={"name": nom})
            if balise and balise.get("content"):
                return norm(balise["content"])
        return ""
    h1 = soupe.find("h1")
    titre = meta("og:title") or (norm(h1.get_text()) if h1 else "") or (norm(soupe.title.string) if soupe.title and soupe.title.string else "")
    titre = titre.split(" | ")[0].strip()
    canon = soupe.find("link", rel="canonical")
    canonique = urljoin(url, canon["href"]).split("#")[0].rstrip("/") if canon and canon.get("href") else ""
    texte, extracteur = "", "bs4"
    if trafilatura is not None:
        try:
            texte = trafilatura.extract(html, url=url, include_comments=False, include_tables=True, favor_recall=True, deduplicate=True) or ""
            extracteur = "trafilatura"
        except Exception:
            texte = ""
    racine = racine_contenu(soupe)
    secours = texte_bs4(racine)
    if len(texte.split()) < MIN_MOTS and len(secours.split()) > len(texte.split()):
        texte, extracteur = secours, "bs4"
    liens = {urljoin(url, a["href"]).split("#")[0].split("?")[0].rstrip("/") for a in (racine.find_all("a", href=True) if racine is not None else [])}
    n_liens = sum(1 for l in liens if urlparse(l).netloc == urlparse(url).netloc and len(parties_chemin(l)) >= 2 and l != url.rstrip("/"))
    return {"titre": titre, "description": meta("description", "og:description"), "canonique": canonique,
            "texte": texte.strip(), "extracteur": extracteur, "n_liens": n_liens}

### 3.7 Tableau brut : une ligne par URL
**Ce que ça fait** : une ligne par URL (même ordre que la liste), avec sa marque, son texte brut, sa langue et sa **source** (« fichier de textes » ou « téléchargé »). `statut` vaut `échec` si la page n'a pas pu être obtenue. On y ajoute le **type de page** : *actualité* (adresse en `/news/`…), *liste* (page qui renvoie vers beaucoup d'articles avec peu de texte), sinon *article*.

In [ ]:
RE_ACTU = re.compile(r"/(news|news-articles|latest-news|press|press-releases?)/", re.IGNORECASE)
lignes = []
for url in urls:
    marque = marque_feuille.get(url) or urlparse(url).netloc.replace("www.", "")
    ligne = {"url": url, "marque": marque, "slug": (parties_chemin(url) or ["index"])[-1], "titre": "", "description": "", "canonique": "",
             "texte": "", "n_liens": 0, "langue": "?", "source": "téléchargé", "statut": "ok", "erreur": erreurs.get(url, "")}
    if url in fournis:
        ligne.update({"titre": fournis[url]["titre"], "description": fournis[url]["description"], "texte": fournis[url]["texte"],
                      "langue": fournis[url]["langue"], "source": "fichier de textes"})
    elif en_cache(url):
        ligne.update(extraire(chemin_cache(url).read_text(encoding="utf-8", errors="replace"), url))
        ligne["langue"] = detecter_langue(ligne["texte"])
    else:
        ligne["statut"] = "échec"
    lignes.append(ligne)
brut = pd.DataFrame(lignes)
brut["n_mots_brut"] = brut["texte"].str.split().str.len().fillna(0).astype(int)
def type_page(r):
    if r["statut"] != "ok":
        return ""
    if RE_ACTU.search(r["url"]):
        return "actualité"
    return "liste" if r["n_liens"] >= 12 and r["n_mots_brut"] / r["n_liens"] < 12 else "article"
brut["type_page"] = brut.apply(type_page, axis=1)
print("brut", brut.shape)
print(brut.groupby(["marque", "type_page"]).size().unstack(fill_value=0).to_string())

### 3.8 Doublons dans une même marque
**Ce que ça fait** : une page est le doublon d'une autre **de la même marque** si elles ont la même adresse canonique (ignorée si plus de 3 pages la partagent : site mal configuré) ou exactement le même texte. Les doublons **entre marques** sont gardés exprès : ce sont des cas de cannibalisation entre marques.

In [ ]:
nb_canoniques = brut.loc[brut["canonique"] != "", "canonique"].value_counts()
brut["empreinte"] = brut["texte"].map(lambda t: hashlib.md5(t.encode()).hexdigest() if t else "")
premiere_url, doublon_de = {}, []
for _, r in brut.iterrows():
    cles = []
    if r["canonique"] and nb_canoniques.get(r["canonique"], 0) <= 3 and len(parties_chemin(r["canonique"])) >= 1:
        cles.append(f"{r['marque']}|c:{r['canonique']}")
    cles.append(f"{r['marque']}|c:" + r["url"].rstrip("/"))
    if r["empreinte"]:
        cles.append(f"{r['marque']}|h:" + r["empreinte"])
    d = next((premiere_url[c] for c in cles if c in premiere_url and premiere_url[c] != r["url"]), "")
    doublon_de.append(d)
    for c in cles:
        premiere_url.setdefault(c, r["url"])
brut["doublon_de"] = doublon_de
print("doublons dans une marque :", int((brut["doublon_de"] != "").sum()))
brut.loc[brut["doublon_de"] != "", ["marque", "url", "doublon_de"]].head(5)

### 3.9 Nettoyage ligne par ligne : outils
**Ce que ça fait** : `premiere_passe` traite chaque page seule : retire le titre répété, les signatures (« Written by », « Share »), les bandeaux de cookies, les noms de produits (signe ™ ou ®, 8 mots au plus) et **coupe tout ce qui suit** « Related articles », « You may also like »… (les « References » ne sont coupées que dans les 40 % de fin de page). Les expressions sont en anglais car les sites le sont.

In [ ]:
RE_SIGNATURE = re.compile(
    r"^(written by|featuring|by [A-Z][a-z]+ [A-Z]|from the team|in this article:?$|share( this)?( article| post)?:?$|reviewed by|medically reviewed|"
    r"last updated|view .{0,40}profile|.{0,30}\(opens in new (window|tab)\)$|we use cookies|.{0,40}cookie (settings|policy|preferences)|"
    r"(strictly necessary|performance|functional|targeting|analytics) cookies$|your privacy$|accept all cookies|manage cookies)", re.IGNORECASE)
RE_COUPE = re.compile(r"^(related articles?|related posts?|related content|you may also like|more from .{0,40}|recommended (articles|reading)|"
                      r"explore more|contextual links|read next)$", re.IGNORECASE)               # tout ce qui suit est retiré
RE_FIN = re.compile(r"^(references|sources|bibliography|contributors?|about the authors?)$", re.IGNORECASE)   # coupé seulement si dans les 40 % de fin
RE_ETIQUETTE = re.compile(r"^(table of contents|contents|sample size: n ?= ?\d+)$", re.IGNORECASE)
RE_MARQUE_DEPOSEE = re.compile("[™®]")

def est_puce(ligne):
    return bool(re.match("^(-|\\*|•|\\d+[.)])\\s", ligne))

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
def premiere_passe(lignes, titre):
    "Renvoie (lignes gardées, [(ligne retirée, raison)])."
    gardees, retirees = [], []
    for i, ligne in enumerate(lignes):
        if RE_COUPE.match(ligne) or (RE_FIN.match(ligne) and i >= 0.6 * len(lignes)):
            retirees += [(l, "fin de page (articles liés, références)") for l in lignes[i:]]
            break
        if RE_FIN.match(ligne) or RE_ETIQUETTE.match(ligne):
            retirees.append((ligne, "étiquette"))
        elif i == 0 and ligne.lower() == norm(titre).lower():
            retirees.append((ligne, "titre répété"))
        elif i > 0 and ligne == lignes[i - 1]:
            retirees.append((ligne, "ligne répétée"))
        elif RE_SIGNATURE.search(re.sub("^[-*\u2022]\\s*", "", ligne)):
            retirees.append((ligne, "signature / bandeau"))
        elif RE_MARQUE_DEPOSEE.search(ligne) and len(ligne.split()) <= 8:
            retirees.append((ligne, "nom de produit"))
        else:
            gardees.append(ligne)
    return gardees, retirees

### 3.10 Nettoyage : blocs répétés dans une marque
**Ce que ça fait** : `nettoyer_marque` retire les lignes qui reviennent dans trop de pages d'une même marque (menus, encarts promo, teasers). Une ligne d'au moins `MOTS_MIN_LIGNE_REPETEE` mots est un bloc répété si elle est dans au moins `PAGES_MIN_REPETITION` pages (`PAGES_MIN_REPETITION_LISTE` pour une puce). **Une ligne partagée par seulement deux pages n'est jamais retirée** : ce recouvrement est justement le signe d'une cannibalisation. Un teaser est conservé dans une seule page : celle dont c'est la description, sinon celle où il apparaît dans les `LIGNES_OUVERTURE` premières lignes.

In [ ]:
def blocs_repetes(docs, pages):
    "Renvoie (lignes répétées, {ligne: page qui la garde}, {ligne: pages où elle apparaît}) pour une marque."
    ou = defaultdict(set)                                   # ligne -> pages où elle apparaît
    for url, lignes in docs.items():
        for l in set(lignes):
            ou[l].add(url)
    repetees = {l for l, us in ou.items() if len(l.split()) >= MOTS_MIN_LIGNE_REPETEE
                and len(us) >= (PAGES_MIN_REPETITION_LISTE if est_puce(l) else PAGES_MIN_REPETITION)}
    descriptions = dict(zip(pages["url"], pages["description"].fillna("").astype(str)))
    proprietaire = {}                                       # bloc répété -> l'unique page qui le garde
    for l in repetees:
        if RE_MARQUE_DEPOSEE.search(l):
            continue                                        # promotion : jamais gardée
        a_pour_description = sorted(u for u in ou[l] if norm(descriptions.get(u, "")).lower() == l.lower())
        if a_pour_description:
            proprietaire[l] = a_pour_description[0]
            continue
        position, url = min((docs[u].index(l), u) for u in ou[l])
        if position < LIGNES_OUVERTURE:
            proprietaire[l] = url
    return repetees, proprietaire, ou

*(suite de la cellule précédente : même étape, découpée pour rester courte)*

In [ ]:
def nettoyer_marque(pages):
    "Renvoie ({url: texte nettoyé}, journal des lignes retirées) pour les pages d'une marque."
    docs, journal = {}, []
    for _, r in pages.iterrows():
        lignes = [norm(l) for l in str(r["texte"]).split("\n") if norm(l)]
        gardees, retirees = premiere_passe(lignes, r["titre"])
        docs[r["url"]] = gardees
        journal += [(r["marque"], r["url"], l, raison) for l, raison in retirees]
    repetees, proprietaire, ou = blocs_repetes(docs, pages)
    propres = {}
    for url, lignes in docs.items():
        retirees = [l for l in lignes if l in repetees and proprietaire.get(l) != url]
        journal += [(pages["marque"].iloc[0], url, l, f"bloc répété ({len(ou[l])} pages)") for l in retirees]
        propres[url] = re.sub(r"\s*\((open|opens) in new (tab|window)\)", "", "\n".join(l for l in lignes if l not in retirees)).strip()
    return propres, journal

### 3.11 Application du nettoyage
**Ce que ça fait** : nettoie **seulement les pages téléchargées**, marque par marque (les textes du fichier `FICHIER_TEXTES` sont déjà nettoyés). Les lignes retirées sont sauvegardées dans `lignes_retirees.csv` pour pouvoir vérifier ce qui a été enlevé. Une page d'article qui garde moins de `MIN_MOTS` mots est marquée *court* ; on lui ajoute sa description (pages vidéo, par exemple) mais elle reste dans le corpus.

In [ ]:
ok = brut[brut["statut"] == "ok"].copy()
propres, journal = {}, []
for _, groupe in ok[ok["source"] == "téléchargé"].groupby("marque"):
    p, j = nettoyer_marque(groupe)
    propres.update(p)
    journal += j
ok["texte_propre"] = [propres.get(u, t) for u, t in zip(ok["url"], ok["texte"])]
ok["n_mots"] = ok["texte_propre"].str.split().str.len().fillna(0).astype(int)
court = (ok["type_page"] == "article") & (ok["n_mots"] < MIN_MOTS)
ok.loc[court, "type_page"] = "court"
ajout = court & ok.apply(lambda r: bool(r["description"]) and r["description"].lower() not in r["texte_propre"].lower(), axis=1)
ok.loc[ajout, "texte_propre"] = (ok.loc[ajout, "description"] + "\n" + ok.loc[ajout, "texte_propre"]).str.strip()
ok["n_mots"] = ok["texte_propre"].str.split().str.len().fillna(0).astype(int)
retirees = pd.DataFrame(journal, columns=["marque", "url", "ligne", "raison"])
retirees.groupby(["marque", "raison", "ligne"]).size().reset_index(name="n_pages").to_csv(f"{OUT_DIR}/carte_urls/lignes_retirees.csv", index=False, encoding="utf-8-sig")
print(f"{len(ok)} pages nettoyées | {len(retirees)} lignes retirées :")
print(retirees["raison"].str.replace(r" \(\d+ pages\)", "", regex=True).value_counts().to_string() if len(retirees) else "aucune")

### 3.12 Corpus final : les articles gardés et les écartés
**Ce que ça fait** : garde les pages de type article (ou court), dans une langue de `LANGUES`, qui ne sont pas des doublons et qui ont du texte. Toutes les autres vont dans le tableau `echecs` **avec leur raison** (rien n'est ignoré en silence). Le tableau `art` est le corpus : **sa ligne i est l'article i** (règle d'alignement pour toute la suite : vecteurs, thèmes, carte).

In [ ]:
def raison_ecart(r):
    "Pourquoi cette URL n'est pas dans le corpus (vide = gardée)."
    if r["statut"] != "ok":
        return "téléchargement impossible : " + (r["erreur"] or "page absente")
    if r["type_page"] in ("actualité", "liste"):
        return f"page de type {r['type_page']}"
    if LANGUES and r["langue"] not in LANGUES:
        return f"langue « {r['langue']} » hors de LANGUES"
    if r["doublon_de"]:
        return "doublon de " + r["doublon_de"]
    return "" if r["n_mots"] > 0 else "aucun texte"

ok = brut.merge(ok[["url", "texte_propre", "n_mots"]], on="url", how="left")
ok["n_mots"] = ok["n_mots"].fillna(0).astype(int)
ok["raison"] = ok.apply(raison_ecart, axis=1)
echecs = ok.loc[ok["raison"] != "", ["url", "raison"]].reset_index(drop=True)
art = ok[ok["raison"] == ""].copy()
art["texte"] = art["texte_propre"]
art = art[["url", "marque", "slug", "titre", "description", "texte", "langue", "source", "type_page", "n_mots"]].reset_index(drop=True)
art["article_id"] = range(len(art))                         # ligne i = article i
n = len(art)
assert n >= 5, f"seulement {n} page(s) exploitable(s) sur {len(urls)} : il en faut au moins 5. Voir le tableau echecs ; fournir les textes dans un CSV (colonnes « url », « titre », « texte ») si les sites refusent les accès."
print("articles", art.shape, "|", art["marque"].value_counts().to_dict(), "|", art["source"].value_counts().to_dict())
print("écartés :", len(echecs), "|", echecs["raison"].str.replace(r"(doublon de|téléchargement impossible :) .*", r"\1", regex=True).value_counts().to_dict())
art[["marque", "titre", "n_mots", "langue", "source"]].head(5)

## 4. Vecteurs (sens de chaque article)
**Ce que ça fait** : reprend l'étape 2 du projet Mars (`mars_step2_one_cell.py`). Chaque article (titre + description + texte) devient 1024 nombres avec bge-m3 : deux articles proches en sens ont des vecteurs proches. Sur GPU (T4), le modèle est chargé en demi-précision (2 fois plus rapide, moitié moins de mémoire) ; si la mémoire manque, **le lot est divisé par 2 tout seul** et on réessaie.
**Paramètres qui comptent** : `MAX_TOKENS` (vitesse contre longueur lue), `TAILLE_LOT`. Sur CPU c'est lent : tester d'abord avec `N_MAX_URLS`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

PERIPHERIQUE = "cuda" if torch.cuda.is_available() else "cpu"
if PERIPHERIQUE == "cpu":
    print("!! pas de GPU : la vectorisation sera lente (Colab : Exécution > Modifier le type d'exécution > GPU T4)")
modele = SentenceTransformer(MODELE, device=PERIPHERIQUE)
if PERIPHERIQUE == "cuda":
    modele.half()                                        # demi-précision : plus rapide, moitié moins de mémoire GPU
print(f"{MODELE} sur {PERIPHERIQUE}, dimension {modele.get_sentence_embedding_dimension()}")

def texte_pour_modele(r):
    "Titre + description (si elle n'est pas déjà le début du texte) + texte, comme à l'étape 2 du projet Mars."
    parties = [r["titre"].strip()]
    d = r["description"].strip()
    if d and d.lower() not in r["texte"][:len(d) + 50].lower():
        parties.append(d)
    parties.append(r["texte"].strip())
    return "\n".join(p for p in parties if p)

def encoder(textes, lot=64, max_tokens=None):
    "Vecteurs normalisés. Textes triés du plus long au plus court ; si le GPU manque de mémoire, le lot est divisé par 2."
    modele.max_seq_length = max_tokens or MAX_TOKENS
    ordre = np.argsort([-len(t) for t in textes])         # les plus longs d'abord : un problème de mémoire apparaît tout de suite
    sortie = np.zeros((len(textes), modele.get_sentence_embedding_dimension()), dtype=np.float32)
    i, n_lots, t0 = 0, 0, time.time()
    while i < len(textes):
        idx = ordre[i:i + lot]
        try:
            vec = modele.encode([textes[j] for j in idx], batch_size=len(idx), normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower() or lot == 1:
                raise
            torch.cuda.empty_cache()
            lot = max(1, lot // 2)
            print(f"  mémoire GPU pleine : lot ramené à {lot}")
            continue
        sortie[idx] = vec.astype(np.float32)
        i += len(idx)
        n_lots += 1
        if i == len(textes) or n_lots % 10 == 0:
            print(f"  {i}/{len(textes)} vectorisés ({time.time() - t0:.0f} s)")
    return sortie

t0 = time.time()
X = encoder([texte_pour_modele(r) for _, r in art.iterrows()], lot=TAILLE_LOT if PERIPHERIQUE == "cuda" else 2, max_tokens=MAX_TOKENS)
assert np.isfinite(X).all(), "valeurs non finies dans les vecteurs"
X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)     # norme 1 : le K-means euclidien équivaut à un regroupement par cosinus
assert len(X) == len(art), "vecteurs et articles doivent avoir le même nombre de lignes"
sim = X @ X.T
print("X :", X.shape, f"en {time.time() - t0:.0f} s | normes {np.linalg.norm(X, axis=1).min():.4f}-{np.linalg.norm(X, axis=1).max():.4f}",
      f"| similarité médiane entre deux articles : {np.median(sim[np.triu_indices(n, 1)]):.3f}")

### 4.1 Vecteurs de passages (facultatif)
**Ce que ça fait** : coupe chaque article en passages d'environ `TAILLE_PASSAGE` mots **sans casser un paragraphe** (un paragraphe trop long est coupé par phrases) et vectorise chaque passage. Ces vecteurs servent à montrer **quels passages** de deux articles se recoupent (cannibalisation). Désactiver avec `CALCULER_PASSAGES = False` pour gagner du temps. `passages.doc_row` renvoie à la ligne de `art`.

In [ ]:
def decouper(texte, taille):
    "Passages d'environ `taille` mots, sans couper un paragraphe (sauf s'il est plus long : coupe par phrases)."
    blocs = []
    for para in [p.strip() for p in texte.split("\n") if p.strip()]:
        if len(para.split()) <= taille:
            blocs.append(para)
            continue
        courant = []
        for phrase in re.split(r"(?<=[.!?])\s+", para):
            if courant and len(" ".join(courant + [phrase]).split()) > taille:
                blocs.append(" ".join(courant))
                courant = []
            courant.append(phrase)
        if courant:
            blocs.append(" ".join(courant))
    passages, courant = [], []
    for b in blocs:
        if courant and len(" ".join(courant + [b]).split()) > taille:
            passages.append("\n".join(courant))
            courant = []
        courant.append(b)
    if courant:
        passages.append("\n".join(courant))
    if len(passages) > 1 and len(passages[-1].split()) < taille // 4:       # dernier passage très court : fusionné avec le précédent
        dernier = passages.pop()
        passages[-1] += "\n" + dernier
    return passages

if CALCULER_PASSAGES:
    passages = pd.DataFrame([{"doc_row": r.Index, "marque": r.marque, "slug": r.slug, "url": r.url, "no": k, "n_mots": len(p.split()), "texte": p}
                             for r in art.itertuples() for k, p in enumerate(decouper(r.texte, TAILLE_PASSAGE))])
    t0 = time.time()
    X_passages = encoder(passages["texte"].tolist(), lot=32 if PERIPHERIQUE == "cuda" else 8, max_tokens=1024)
    assert np.isfinite(X_passages).all() and len(X_passages) == len(passages)
    print(f"passages : {passages.shape}, vecteurs {X_passages.shape} en {time.time() - t0:.0f} s | médiane {passages['n_mots'].median():.0f} mots")
    passages.head(3)
else:
    passages, X_passages = None, None
    print("passages non calculés (CALCULER_PASSAGES = False)")

## 5. Thèmes (K-means)
**Ce que ça fait** : regroupe les articles en `K` thèmes (par défaut racine de n/2, entre 3 et 20). Pour chaque article on garde la silhouette (1 = bien rangé, 0 = à la frontière)
et le thème voisin le plus proche. **Paramètres qui comptent** : `K`, `GRAINE`. Le bon nombre de thèmes n'est pas « calculé » : essayer plusieurs `K` et regarder ce qui est lisible.

In [ ]:
K_FINAL = K if K else int(np.clip(round(np.sqrt(n / 2)), 3, 20))
kmeans = KMeans(n_clusters=K_FINAL, n_init=N_INIT, max_iter=MAX_ITER, random_state=GRAINE).fit(X)
theme = kmeans.labels_
centres = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, axis=1, keepdims=True)
distance_centre = 1 - np.sum(X * centres[theme], axis=1)       # 1 - cosinus entre l'article et son centre
sims_centres = X @ centres.T
sims_centres[np.arange(n), theme] = -np.inf                    # on exclut son propre thème
theme_voisin = sims_centres.argmax(axis=1)
silhouette = silhouette_samples(X, theme, metric="cosine")
print(f"{K_FINAL} thèmes | articles par thème : {np.bincount(theme).tolist()} | silhouette moyenne {silhouette.mean():.3f}")

## 6. Noms des thèmes
**Idée** (la même que dans le notebook Mars) : nommer un thème avec les 2 expressions de 1 à 3 mots qui le résument le mieux et le distinguent des autres.
Candidates = expressions présentes dans au moins `COUVERTURE_MIN` des articles du thème ; score = ressemblance (bge-m3) avec le centre du thème − `POIDS_DISTINCTION` × ressemblance avec le thème voisin + `POIDS_COUVERTURE` × couverture ;
la 2e expression doit être différente de la 1re. Les mots vides anglais et les noms de domaine des sites sont ignorés.

In [ ]:
mots_domaines = {m for site in art["marque"].unique() for m in re.findall(r"[a-z]{3,}", site.lower())}   # ex. iams, cesar : pas des noms de thèmes
MOTS_VIDES = list(ENGLISH_STOP_WORDS | {"also", "may", "can", "will", "one", "just", "like", "get", "make"} | mots_domaines | set(MOTS_VIDES_SUPPL))
MOTS_GENERAUX = set()      # mots trop généraux pour nommer un thème seuls (vide ici : à remplir selon le sujet du corpus)

VECTEUR_MOTS = dict(token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b", stop_words=MOTS_VIDES)

def mots_cles_par_theme(textes, labels, k):
    """c-TF-IDF : un « document » par thème = concaténation de ses articles. Renvoie N_MOTS_CLES mots par thème."""
    docs = [" ".join(textes[labels == c]) for c in range(k)]
    vect = CountVectorizer(ngram_range=(1, 2), min_df=1, **VECTEUR_MOTS)
    comptes = vect.fit_transform(docs).toarray().astype(float)
    tf = comptes / np.clip(comptes.sum(axis=1, keepdims=True), 1, None)
    idf = np.log(1 + (comptes.sum() / k) / np.clip(comptes.sum(axis=0), 1, None))
    vocab = np.array(vect.get_feature_names_out())
    return [vocab[np.argsort(-(tf * idf)[c])[:N_MOTS_CLES]].tolist() for c in range(k)]

def candidates_par_theme(textes, labels, k):
    """Expressions de 1 à 3 mots assez présentes dans le thème : [(expression, couverture)] par thème."""
    vect = CountVectorizer(ngram_range=(1, 3), min_df=2, binary=True, **VECTEUR_MOTS)
    presence = vect.fit_transform(textes)                                  # article x expression (0/1)
    vocab = np.array(vect.get_feature_names_out())
    par_theme = np.vstack([np.asarray(presence[labels == c].sum(axis=0)).ravel() for c in range(k)])
    couverture = par_theme / np.bincount(labels, minlength=k)[:, None]     # part des articles du thème
    idf = np.log(1 + par_theme.sum() / k / np.clip(par_theme.sum(axis=0), 1, None))
    general = np.array([set(e.split()) <= MOTS_GENERAUX for e in vocab])
    score = couverture * idf
    resultat = []
    for c in range(k):
        ok = np.where((couverture[c] >= COUVERTURE_MIN) & ~general)[0]
        ordre = ok[np.argsort(-score[c, ok])][:N_CANDIDATS]
        resultat.append([(vocab[i], float(couverture[c, i])) for i in ordre])
    return resultat

def racines(expression):
    """Mots sans le pluriel simple (dogs -> dog), pour éviter « Dog · Dogs »."""
    return {m[:-1] if m.endswith("s") and len(m) > 3 else m for m in expression.lower().split()}

def centres_unitaires(Xc, labels, k):
    c = np.stack([Xc[labels == i].mean(axis=0) for i in range(k)])
    return c / np.linalg.norm(c, axis=1, keepdims=True)

def choisir_expressions(Xc, labels, k, candidates):
    """Pour chaque thème, les 2 expressions retenues (liste de listes)."""
    ctr = centres_unitaires(Xc, labels, k)
    uniques = sorted({e for liste in candidates for e, _ in liste})
    vec = dict(zip(uniques, encoder(uniques))) if uniques else {}
    choisies = []
    for c in range(k):
        liste = candidates[c]
        if not liste:
            choisies.append([]); continue
        V = np.stack([vec[e] for e, _ in liste])
        couv = np.array([cv for _, cv in liste])
        autres = np.delete(np.arange(k), c)
        voisin = (V @ ctr[autres].T).max(axis=1)                  # ressemblance avec l'autre thème le plus proche
        score = V @ ctr[c] - POIDS_DISTINCTION * voisin + POIDS_COUVERTURE * couv
        premier = int(np.argmax(score))
        mmr = 0.7 * score - 0.3 * (V @ V[premier])                # 2e : bon score ET différente de la 1re
        for i, (e, _) in enumerate(liste):
            if i == premier or racines(e) & racines(liste[premier][0]):
                mmr[i] = -np.inf
        choisies.append([liste[premier][0]] + ([liste[int(np.argmax(mmr))][0]] if np.isfinite(mmr).any() else []))
    return choisies

**Application** : textes = titre (compté 2 fois) + texte. Nom d'un thème = les 2 expressions en majuscule initiale, séparées par « · » ; à défaut les mots-clés les plus caractéristiques.

In [ ]:
textes_themes = (art["titre"] + " " + art["titre"] + " " + art["texte"]).to_numpy()
mots_cles = mots_cles_par_theme(textes_themes, theme, K_FINAL)
expressions = choisir_expressions(X, theme, K_FINAL, candidates_par_theme(textes_themes, theme, K_FINAL))
theme_nom = [" · ".join(e.capitalize() for e in expr) or " / ".join(mots_cles[c][:3]) for c, expr in enumerate(expressions)]
pd.DataFrame({"theme": range(K_FINAL), "nom": theme_nom, "n_articles": np.bincount(theme)})

## 7. Carte 2D (UMAP)
**Ce que ça fait** : projette les vecteurs en 2 dimensions avec UMAP (distance cosinus). **Attention** : une carte 2D déforme les distances ; ne jamais conclure à une lacune de contenu à partir d'une zone vide.
La *fidélité* (trustworthiness, de 0 à 1) mesure la part des voisins de la carte qui sont de vrais voisins dans l'espace complet. **Paramètres qui comptent** : `UMAP_VOISINS`, `UMAP_MIN_DIST`, `GRAINE`.

In [ ]:
coords = umap.UMAP(n_components=2, metric="cosine", n_neighbors=min(UMAP_VOISINS, n - 1),
                   min_dist=UMAP_MIN_DIST, random_state=GRAINE, n_jobs=1).fit_transform(X)
fidelite = round(float(trustworthiness(X, coords, n_neighbors=min(N_VOISINS_CONFIANCE, n // 2 - 1), metric="cosine")), 3)
print("coords", coords.shape, "| fidélité UMAP :", fidelite)

## 8. Tableaux de résultats
**Ce que ça fait** : un tableau par article (`articles`), un par thème (`resume_themes`), et la liste des `PAIRES_AFFICHEES` paires d'articles les plus proches en sens (`proches`).
Une paire très proche = deux pages qui répondent probablement à la même question : à lire avant d'en conclure quoi que ce soit (une similarité élevée ne prouve pas la même intention de recherche).

In [ ]:
articles = art[["article_id", "url", "marque", "langue", "titre", "slug", "n_mots", "source"]].copy()
articles["theme"], articles["theme_nom"] = theme, [theme_nom[c] for c in theme]
articles["voisin_nom"] = [theme_nom[c] for c in theme_voisin]
articles["silhouette"] = silhouette.round(4)
articles["distance_centre"] = distance_centre.round(4)
articles["rang_dans_theme"] = articles.groupby("theme")["distance_centre"].rank(method="first").astype(int)
articles["umap_x"], articles["umap_y"] = coords[:, 0].round(4), coords[:, 1].round(4)
articles["description"] = art["description"].str.slice(0, 400)
lignes = []
for c in range(K_FINAL):
    membres = np.where(theme == c)[0]
    central = membres[np.argmax(X[membres] @ centres[c])]                    # l'article le plus proche du centre du thème
    lignes.append({"theme": c, "nom": theme_nom[c], "n_articles": len(membres), "silhouette_moyenne": round(float(silhouette[membres].mean()), 3),
                   "titre_representatif": art.at[central, "titre"], "mots_cles": ", ".join(mots_cles[c]),
                   "sites": ", ".join(f"{m} {k}" for m, k in art.loc[membres, "marque"].value_counts().items())})
resume_themes = pd.DataFrame(lignes)
print(articles.shape, resume_themes.shape)
resume_themes[["theme", "nom", "n_articles", "titre_representatif"]]

iu, ju = np.triu_indices(n, 1)
meilleures = np.argsort(-sim[iu, ju])[:PAIRES_AFFICHEES]
a, b = iu[meilleures], ju[meilleures]
proches = pd.DataFrame({"similarite": sim[a, b].round(3), "site_a": art["marque"].values[a], "titre_a": art["titre"].str.slice(0, 60).values[a],
                        "site_b": art["marque"].values[b], "titre_b": art["titre"].str.slice(0, 60).values[b],
                        "url_a": art["url"].values[a], "url_b": art["url"].values[b], "meme_site": art["marque"].values[a] == art["marque"].values[b]})
print(proches.shape, "| dont même site :", int(proches["meme_site"].sum()))
proches[["similarite", "site_a", "titre_a", "site_b", "titre_b"]].head(8)

articles.drop(columns=["description"]).to_csv(f"{OUT_DIR}/carte_urls/articles.csv", index=False, encoding="utf-8-sig")
resume_themes.to_csv(f"{OUT_DIR}/carte_urls/themes.csv", index=False, encoding="utf-8-sig")
proches.to_csv(f"{OUT_DIR}/carte_urls/paires_proches.csv", index=False, encoding="utf-8-sig")
echecs.to_csv(f"{OUT_DIR}/carte_urls/pages_en_echec.csv", index=False, encoding="utf-8-sig")
print(sorted(os.listdir(f"{OUT_DIR}/carte_urls")))

---
# Annexe — affichage : la carte interactive
**Cette section n'est pas du calcul** : elle fabrique une page HTML autonome (carte Plotly + fiche d'article au clic + recherche) à partir du tableau `articles`.
Le HTML, le CSS et le JavaScript sont dans des chaînes de texte, volontairement peu détaillés.

In [ ]:
COULEURS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
            "#393b79", "#ad494a", "#637939", "#8c6d31", "#843c39", "#7b4173", "#3182bd", "#e6550d", "#31a354", "#756bb1"]
COULEURS_FIXES = {"dog": "#d97706", "cat": "#2563eb", "dog and cat": "#0f9d76", "unspecified": "#9ca3af", "dry": "#b45309",
                  "wet": "#0ea5e9", "treats": "#16a34a", "not food-specific": "#cbd5e1", "puppy / kitten": "#f59e0b",
                  "adult": "#2563eb", "senior": "#7c3aed", "all ages": "#cbd5e1", "yes": "#dc2626", "no": "#cbd5e1", "": "#e2e8f0"}

def surface(xy):
    return float(np.ptp(xy[:, 0]) * np.ptp(xy[:, 1])) or None        # aire du rectangle qui contient la carte

def zone_theme(xy, aire):
    """Contour des PART_ZONE articles les plus proches du centre du thème (None si trop petit ou dispersé)."""
    if len(xy) < 6:
        return None
    d = np.linalg.norm(xy - np.median(xy, axis=0), axis=1)
    garde = xy[d <= np.quantile(d, PART_ZONE)]
    try:
        hull = ConvexHull(garde)
    except Exception:
        return None                                                    # points alignés : pas de zone
    if aire and hull.volume > ZONE_MAX * aire:                         # en 2D, volume = aire
        return None
    pts = garde[hull.vertices]
    return np.vstack([pts, pts[:1]])

MODES = [("theme", "theme", "Couleur : thème"), ("marque", "marque", "Couleur : site"), ("langue", "langue", "Couleur : langue")]

def traces_zones(a):
    traces, aire = [], surface(a[["umap_x", "umap_y"]].to_numpy())
    for c in sorted(a["theme"].unique()):
        z = zone_theme(a.loc[a["theme"] == c, ["umap_x", "umap_y"]].to_numpy(), aire)
        if z is not None:
            couleur = COULEURS[c % len(COULEURS)]
            traces.append({"type": "scatter", "x": z[:, 0].round(4).tolist(), "y": z[:, 1].round(4).tolist(), "mode": "lines",
                           "fill": "toself", "fillcolor": couleur, "opacity": 0.13, "line": {"color": couleur, "width": 1},
                           "hoverinfo": "skip", "showlegend": False})
    return traces

def texte_survol(d):
    return (d["titre"].str.slice(0, 90) + "<br>" + d["marque"] + " · thème " + d["theme"].astype(str) + " · " + d["theme_nom"]).tolist()

def ajouter_points(a, traces):
    "Ajoute un groupe de traces par mode de couleur ; renvoie {mode: indices des traces}."
    visibles = {m: [] for m, _, _ in MODES}
    for mode, col, _ in MODES:
        for i, g in enumerate(sorted(a[col].unique())):
            d = a[a[col] == g]
            nom = f"{g} · {d['theme_nom'].iloc[0]}" if mode == "theme" else str(g)
            couleur = COULEURS[int(g) % len(COULEURS)] if mode == "theme" else COULEURS[(i * 3) % len(COULEURS)]
            traces.append({"type": "scatter", "mode": "markers", "x": d["umap_x"].tolist(), "y": d["umap_y"].tolist(),
                           "name": f"{nom[:55]} — {len(d)} ({100 * len(d) / len(a):.1f}%)", "visible": mode == "theme", "legendgroup": mode,
                           "marker": {"size": 9, "opacity": 0.85, "color": couleur, "line": {"width": 0.5, "color": "white"}},
                           "customdata": d["article_id"].tolist(), "hovertemplate": "%{text}<extra></extra>", "text": texte_survol(d)})
            visibles[mode].append(len(traces) - 1)
    return visibles

def annotations_themes(a):
    return [{"x": float(g["umap_x"].median()), "y": float(g["umap_y"].median()), "showarrow": False,
             "text": f"<b>{c}</b> {g['theme_nom'].iloc[0][:28]}", "font": {"size": 11, "color": COULEURS[int(c) % len(COULEURS)]},
             "bgcolor": "rgba(255,255,255,0.75)", "borderpad": 2} for c, g in a.groupby("theme")]

def figure_carte(a, titre):
    traces = traces_zones(a)
    zones = list(range(len(traces)))
    visibles = ajouter_points(a, traces)
    points = sum(visibles.values(), [])
    traces.append({"type": "scatter", "mode": "markers", "x": [], "y": [], "name": "recherche", "hoverinfo": "skip",
                   "showlegend": False, "marker": {"size": 17, "symbol": "circle-open", "color": "black", "line": {"width": 2.5}}})
    boutons = [{"label": lab, "method": "restyle", "args": [{"visible": [i in visibles[m] for i in points]}, points]} for m, _, lab in MODES]
    menu_couleurs = {"buttons": boutons, "direction": "down", "x": 0, "y": 1.1, "xanchor": "left"}
    menu_zones = {"buttons": [{"label": "Zones : oui", "method": "restyle", "args": [{"visible": True}, zones]},
                              {"label": "Zones : non", "method": "restyle", "args": [{"visible": False}, zones]}],
                  "direction": "down", "x": 0.25, "y": 1.1, "xanchor": "left"}
    axe = {"showgrid": False, "zeroline": False}
    layout = {"title": {"text": titre}, "height": 760, "template": "plotly_white", "hovermode": "closest", "dragmode": "pan", "clickmode": "event",
              "xaxis": {"title": "UMAP 1", **axe}, "yaxis": {"title": "UMAP 2", **axe}, "annotations": annotations_themes(a),
              "legend": {"title": {"text": "Articles (part du total) · cliquer pour masquer"}, "font": {"size": 10}},
              "updatemenus": [menu_couleurs, menu_zones]}
    for i in range(len(traces)):                          # zones visibles au départ, comme les points « thème » (déjà réglés)
        if i in zones or i == len(traces) - 1:
            traces[i]["visible"] = True
    return {"data": traces, "layout": layout}

STYLE = """<style>
body { font-family: Arial, sans-serif; margin: 24px; color: #222; background: #fff; }
.aide { color: #555; }
.panneau { display: grid; grid-template-columns: minmax(280px, 1fr) minmax(280px, 1.3fr); gap: 16px; margin: 6px 0 40px; }
@media (max-width: 900px) { .panneau { grid-template-columns: 1fr; } }
.fiche, .liste { border: 1px solid #ccc; border-radius: 8px; padding: 12px 16px; background: #f7f7f7; line-height: 1.6; }
.fiche h3, .liste h3 { margin: 0 0 8px; }
.fiche table { border-collapse: collapse; width: 100%; }
.fiche td { padding: 3px 6px; vertical-align: top; border-bottom: 1px solid #e3e3e3; overflow-wrap: anywhere; }
.fiche td:first-child { color: #555; white-space: nowrap; width: 1%; }
.liste ol { max-height: 360px; overflow-y: auto; margin: 0; padding-left: 22px; }
.liste li { margin: 3px 0; cursor: pointer; }
.liste li:hover { text-decoration: underline; }
.liste li.actif { font-weight: bold; }
.pastille { display: inline-block; width: 12px; height: 12px; border-radius: 3px; margin-right: 6px; vertical-align: -1px; }
a { color: #155eef; }
.recherche { margin: 10px 0 4px; }
.recherche input { width: min(560px, 90%); padding: 7px 10px; font-size: 14px; border: 1px solid #bbb; border-radius: 6px; }
.vues { margin: 8px 0 2px; }
.vue { font-size: 14px; padding: 6px 12px; margin-right: 6px; border: 1px solid #bbb; border-radius: 6px; background: #fff; cursor: pointer; }
.vue.actif { background: #1f2933; color: #fff; border-color: #1f2933; }
</style>"""

JS_OUTILS = r"""<script>
function esc(s) { return String(s ?? "").replace(/[&<>"']/g, m => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[m])); }
function ligne(nom, valeur) {
  if (valeur === null || valeur === undefined || valeur === "") return "";
  return "<tr><td>" + esc(nom) + "</td><td>" + valeur + "</td></tr>";
}
function ajuster() {
  try { const cadre = window.frameElement; if (cadre) cadre.style.height = (document.documentElement.scrollHeight + 20) + "px"; } catch (e) {}
}
function vue(divId, nom) {
  const gd = document.getElementById(divId);
  if (!gd || !gd.layout || !gd.layout.meta) return;
  const V = gd.layout.meta.vues[nom];
  Plotly.restyle(gd, {visible: V.visible});
  Plotly.relayout(gd, {updatemenus: V.updatemenus, annotations: V.annotations});
  ["themes", "mars"].forEach(n => document.getElementById(divId + "-vue-" + n).classList.toggle("actif", n === nom));
}
</script>"""

JS_FICHE = r"""<script>
function afficher(divId, id) {
  const A = ARTICLES[id];
  if (!A) return;
  const couleur = COULEURS_THEMES[A.theme] || "#999";
  document.getElementById(divId + "-fiche").innerHTML = "<h3>Article n°" + esc(A.article_id) + "</h3><table>" +
    ligne("Titre", "<b>" + esc(A.titre) + "</b>") + ligne("Site", esc(A.marque)) + ligne("Langue", esc(A.langue)) +
    ligne("Lien", '<a href="' + esc(A.url) + '" target="_blank" rel="noopener">' + esc(A.url) + "</a>") +
    ligne("Thème", '<span class="pastille" style="background:' + couleur + '"></span>' + esc(A.theme) + " · " + esc(A.theme_nom)) +
    ligne("Rang dans le thème", esc(A.rang_dans_theme) + " / " + esc(TAILLES[A.theme]) + " (1 = le plus central)") +
    ligne("Silhouette", Number(A.silhouette).toFixed(2) + " (thème voisin : " + esc(A.voisin_nom) + ")") +
    ligne("Mots", esc(A.n_mots)) + ligne("Description", esc(A.description)) + "</table>";
  const membres = ARTICLES.filter(x => x.theme === A.theme).sort((a, b) => a.rang_dans_theme - b.rang_dans_theme);
  let liste = "<h3>Thème " + esc(A.theme) + " · " + esc(A.theme_nom) + " — " + membres.length + " articles (" +
    (100 * membres.length / ARTICLES.length).toFixed(1) + "%)</h3><p class='aide'>Du plus central au plus éloigné. Cliquer sur un titre pour voir sa fiche.</p><ol>";
  membres.forEach(m => {
    liste += '<li class="' + (m.article_id === A.article_id ? "actif" : "") + '" onclick="afficher(\'' + divId + "', " +
      m.article_id + ')">' + esc(m.titre) + " <span class='aide'>(" + esc(m.marque) + ")</span></li>";
  });
  document.getElementById(divId + "-liste").innerHTML = liste + "</ol>";
  ajuster();
}
</script>"""

JS_RECHERCHE = r"""<script>
function rechercher(divId) {
  const texte = document.getElementById(divId + "-recherche").value.trim().toLowerCase();
  const termes = texte.split(/[\s,;]+/).filter(m => m.length >= 2);
  const trouves = termes.length ? ARTICLES.filter(a => {
    const t = (a.titre + " " + a.slug + " " + (a.description || "")).toLowerCase();
    return termes.some(m => t.includes(m));
  }) : [];
  const gd = document.getElementById(divId);
  if (gd && gd.data) Plotly.restyle(gd, {x: [trouves.map(a => a.umap_x)], y: [trouves.map(a => a.umap_y)]}, [gd.data.length - 1]);
  const info = document.getElementById(divId + "-resultat");
  if (!termes.length) { info.textContent = ""; return; }
  info.textContent = trouves.length + " articles";
  let liste = "<h3>Recherche « " + esc(texte) + " » — " + trouves.length + " articles</h3>" +
    "<p class='aide'>Entourés en noir sur la carte. Cliquer sur un titre pour voir sa fiche.</p><ol>";
  trouves.forEach(m => {
    liste += '<li onclick="afficher(\'' + divId + "', " + m.article_id + ')">' + esc(m.titre) +
      " <span class='aide'>(" + esc(m.marque) + " · thème " + esc(m.theme) + ")</span></li>";
  });
  document.getElementById(divId + "-liste").innerHTML = liste + "</ol>";
  ajuster();
}
</script>"""

JS_DESSIN = r"""<script>
function dessiner(divId, figure, essai) {
  if (typeof Plotly === "undefined") {
    if ((essai || 0) < 100) setTimeout(() => dessiner(divId, figure, (essai || 0) + 1), 100);
    else document.getElementById(divId).innerHTML = "<p class='aide'>plotly.js n'a pas pu être chargé : ouvrir le fichier HTML dans un navigateur.</p>";
    return;
  }
  Plotly.newPlot(divId, figure.data, figure.layout, {scrollZoom: true, responsive: true, displaylogo: false})
    .then(gd => gd.on("plotly_click", ev => {
      const p = ev.points && ev.points[0];
      if (p && p.customdata !== undefined && p.customdata !== null) afficher(divId, p.customdata);
    })).then(ajuster);
}
</script>"""

**Assemblage de la page** et enregistrement dans `outputs/carte_urls/carte_urls.html` (autonome : plotly.js est intégré, ouverture hors connexion possible). Dans le notebook on affiche la même page avec plotly.js chargé depuis Internet (notebook plus léger).

In [ ]:
COLONNES_PAGE = ["article_id", "slug", "marque", "langue", "titre", "url", "theme", "theme_nom", "silhouette", "voisin_nom",
                 "rang_dans_theme", "n_mots", "description", "umap_x", "umap_y"]

def construire_page(a, plotly_integre=True):
    def js(objet):                                              # JSON sûr dans une balise <script>
        return json.dumps(objet, ensure_ascii=False, default=str).replace("</", "<\\/")
    donnees = a.sort_values("article_id")[COLONNES_PAGE].replace({np.nan: None}).to_dict(orient="records")
    couleurs = {int(c): COULEURS[int(c) % len(COULEURS)] for c in a["theme"].unique()}
    tailles = {int(c): int(v) for c, v in a["theme"].value_counts().items()}
    script_donnees = f"<script>const ARTICLES = {js(donnees)};\nconst COULEURS_THEMES = {js(couleurs)};\nconst TAILLES = {js(tailles)};</script>"
    script_plotly = ("<script>" + get_plotlyjs() + "</script>") if plotly_integre else "<script src='https://cdn.plot.ly/plotly-2.35.2.min.js'></script>"
    d, titre = "carte_urls", f"Carte UMAP — fidélité {fidelite}"
    corps = (f"<h1>Carte sémantique — {a['theme'].nunique()} thèmes, {len(a)} articles, {a['marque'].nunique()} site(s)</h1>"
             "<p class='aide'>Molette : zoom · glisser : déplacer · double-clic : vue complète · survol : titre · clic : fiche de l'article et articles du même thème · "
             "menus : couleurs et zones. Les axes d'UMAP n'ont pas de signification : une zone vide n'est pas une lacune de contenu.</p>"
             f"<div class='recherche'><input id='{d}-recherche' type='search' placeholder='Chercher dans les titres' "
             f"oninput=\"rechercher('{d}')\"> <span id='{d}-resultat' class='aide'></span></div><div id='{d}'></div>"
             f"<div class='panneau'><div class='fiche' id='{d}-fiche'>Cliquer sur un point pour voir la fiche de l'article.</div>"
             f"<div class='liste' id='{d}-liste'></div></div><script>dessiner('{d}', {js(figure_carte(a, titre))});</script>")
    return ("<!DOCTYPE html><html lang='fr'><head><meta charset='utf-8'><title>Carte sémantique</title>" + script_plotly
            + STYLE + script_donnees + JS_OUTILS + JS_FICHE + JS_RECHERCHE + JS_DESSIN + "</head><body>" + corps + "</body></html>")

chemin_carte = f"{OUT_DIR}/carte_urls/carte_urls.html"
with open(chemin_carte, "w", encoding="utf-8") as f:
    f.write(construire_page(articles, plotly_integre=True))
print(f"Carte enregistrée : {chemin_carte} ({os.path.getsize(chemin_carte) / 1e6:.1f} Mo)")
if IN_COLAB:                           # Colab efface /content en fin de session : on télécharge un zip des résultats
    import zipfile
    from google.colab import files
    with zipfile.ZipFile("/content/carte_urls.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for nom in os.listdir(f"{OUT_DIR}/carte_urls"):
            z.write(f"{OUT_DIR}/carte_urls/{nom}", nom)
    files.download("/content/carte_urls.zip")
from IPython.display import HTML, display
page = construire_page(articles, plotly_integre=False)
display(HTML('<iframe srcdoc="' + html.escape(page, quote=True) + '" style="width:100%; height:2300px; border:0; background:#fff"></iframe>'))